How does revenue change each month?
Which 10 products generate the most revenue?
How much does a customer typically spend per order?
How does spending differ across key countries?

In [ ]:
import pandas as pd
df = pd.read_excel("/Users/juma/Desktop/dATA-VIS-CA2/Online-Retail.xlsx")
print(df.head())             
print(df.shape)              
df.info()                    
print(df.isna().sum())       
print(df.duplicated().sum())

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
(541909, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------      

Shape: 541,909 rows × 8 columns. Each row is one line item on an invoice.
Dtypes: All fine. InvoiceDate is already a datetime. CustomerID is float64 
Missing values:
Description: 1,454 missing (0.3%). Minor.
CustomerID: 135,080 missing (about 25%).
Duplicates: 5,268 fully identical rows (about 1%). 

Quantity: min -80,995, max 80,995. That's one huge order and its cancellation (the famous pair in this dataset). The median is 3, so these are extreme outliers.
UnitPrice: min -11,062.06 is not a real price, it's a bad-debt adjustment. Max 38,970 is likely a manual/AMAZONFEE-type entry, not a product.
10,624 rows with negative Quantity (returns, cancellations, write-offs).
2,517 rows with UnitPrice <= 0 (free items, adjustments).
9,288 rows on C invoices (cancellations). Note 10,624 > 9,288, so about 1,300 negative-quantity rows are not C invoices.


In [10]:
print(df.shape)
print(df.dtypes)
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
print("Negative quantities:", (df['Quantity'] < 0).sum())
print("Zero or negative prices:", (df['UnitPrice'] <= 0).sum())
print("Cancelled invoices:", df['InvoiceNo'].astype(str).str.startswith('C').sum())

# 2. Convert the invoice date
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 3. Clean (print the shape after each step)
clean = df.drop_duplicates()
print("After duplicates:", clean.shape)

# Cancelled invoices start with C
clean = clean[~clean['InvoiceNo'].astype(str).str.startswith('C')]
print("After cancellations:", clean.shape)

# Negative or zero quantity
clean = clean[clean['Quantity'] > 0]
print("After quantity check:", clean.shape)

# Invalid prices
clean = clean[clean['UnitPrice'] > 0]
print("After price check:", clean.shape)

# Non-product codes (postage, fees, manual entries)
codes = ['POST', 'D', 'M', 'BANK CHARGES', 'AMAZONFEE', 'DOT', 'CRUK', 'S', 'B']
clean = clean[~clean['StockCode'].isin(codes)]
print("After non-product codes:", clean.shape)

# 4. New columns
clean['Revenue'] = clean['Quantity'] * clean['UnitPrice']
clean['Month'] = clean['InvoiceDate'].dt.to_period('M').astype(str)

# 5. Final check
print(clean.shape)
print(clean[['Quantity', 'UnitPrice', 'Revenue']].describe())

(541909, 8)
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
Duplicates: 5268
Negative quantities: 10624
Zero or negative prices: 2517
Cancelled invoices: 9288
After duplicates: (536641, 8)
After cancellations: (527390, 8)
After quantity check: (526054, 8)
After price check: (524878, 8)
After non-product codes: (522713, 8)
(522713, 10)
            Quantity      UnitPrice        Revenue
count  522713.000000  522713.000000  522713.000000
mean       10.639802       3.287619      19.618718
std       156.568305       4.464488     269.893940
min         1.000000       0.001000       0.001000
25%         1

In total I removed 19,196 rows (3.5%), so most of the data is kept. Cancellations are the biggest group after duplicates.

                removed          left
Start		                     541,909
Duplicates	    5,268	         536,641
Cancellations(C)9,251	         527,390
Quantity <= 0	1,336	         526,054
Price <= 0	    1,176	         524,878
Non-product codes2,165	         522,713


In [11]:
print(clean.nlargest(5, 'Quantity')[['InvoiceNo', 'Description', 'Quantity', 'UnitPrice', 'InvoiceDate']])

       InvoiceNo                        Description  Quantity  UnitPrice  \
540421    581483        PAPER CRAFT , LITTLE BIRDIE     80995       2.08   
61619     541431     MEDIUM CERAMIC TOP STORAGE JAR     74215       1.04   
421632    573008  WORLD WAR 2 GLIDERS ASSTD DESIGNS      4800       0.21   
206121    554868               SMALL POPCORN HOLDER      4300       0.72   
97432     544612              EMPIRE DESIGN ROSETTE      3906       0.82   

               InvoiceDate  
540421 2011-12-09 09:15:00  
61619  2011-01-18 10:01:00  
421632 2011-10-27 12:26:00  
206121 2011-05-27 10:52:00  
97432  2011-02-22 10:43:00  


Two very large orders 74,215 and 80,995 units

In [12]:
print(df[df['Quantity'].abs() > 70000][['InvoiceNo', 'Description', 'Quantity', 'InvoiceDate']])

       InvoiceNo                     Description  Quantity         InvoiceDate
61619     541431  MEDIUM CERAMIC TOP STORAGE JAR     74215 2011-01-18 10:01:00
61624    C541433  MEDIUM CERAMIC TOP STORAGE JAR    -74215 2011-01-18 10:17:00
540421    581483     PAPER CRAFT , LITTLE BIRDIE     80995 2011-12-09 09:15:00
540422   C581484     PAPER CRAFT , LITTLE BIRDIE    -80995 2011-12-09 09:27:00


both bulk orders were cancelled within about 15 minutes by a C invoice with the exact same quantity, so removing them is justified.
541431 → C541433 (74,215 jars, 18 Jan 2011)
581483 → C581484 (80,995 paper craft items, 9 Dec 2011)


In [13]:
print(clean[clean['UnitPrice'] < 0.1][['Description', 'UnitPrice']].head(10))
print((clean['UnitPrice'] < 0.1).sum())

                                Description  UnitPrice
14788        PORCELAIN BUDAH INCENSE HOLDER      0.070
91772   ESSENTIAL BALM 3.5g TIN IN ENVELOPE      0.060
112953       BLUE STONES ON WIRE FOR CANDLE      0.080
157195           PADS TO MATCH ALL CUSHIONS      0.001
170361       PORCELAIN BUDAH INCENSE HOLDER      0.070
170392  SMALL FOLDING SCISSOR(POINTED EDGE)      0.080
170957          PAPER POCKET TRAVELING FAN       0.070
177435       BLUE STONES ON WIRE FOR CANDLE      0.080
195806       BLUE STONES ON WIRE FOR CANDLE      0.080
210605          PAPER POCKET TRAVELING FAN       0.070
235


In [14]:
# Remove the two cancelled bulk orders
clean = clean[clean['Quantity'] < 70000]
print("After removing cancelled bulk orders:", clean.shape)

After removing cancelled bulk orders: (522711, 10)


Two very large orders (74,215 and 80,995 units) were removed because each was cancelled the same day, so keeping them would overstate revenue